# 01 Data audit

I check the raw price, demand and solar data before trusting any of it. Every number in `docs/data-dictionary.md` comes from this notebook.

In [1]:
import os
from pathlib import Path

import pandas as pd

if Path.cwd().name == 'notebooks':
    os.chdir('..')

from src.fetch_aemo import load_months
from src.profiling import interval_gaps, iqr_outliers, profile_table, reconcile

pd.set_option('display.width', 160)

## AEMO download

AEMO stamps each row with the end of its interval. Files are 30-minute until 5-minute settlement started on 1 Oct 2021.

In [2]:
aemo = load_months('NSW1', '2021-07', '2024-06')
aemo['SETTLEMENTDATE'] = pd.to_datetime(aemo['SETTLEMENTDATE'], format='%Y/%m/%d %H:%M:%S')
print(aemo.shape)
profile_table(aemo)

(293568, 5)


,column,dtype,nulls,null_pct,distinct,min,median,max
0,REGION,str,0,0.0,1,None,NaN,None
1,SETTLEMENTDATE,datetime64[us],0,0.0,293568,2021-07-01 00:30:00,NaN,2024-07-01 00:00:00
2,TOTALDEMAND,float64,0,0.0,215836,3664.34,7380.765,13763.96
3,RRP,float64,0,0.0,40309,-1000.0,89.000,16600.0
4,PERIODTYPE,str,0,0.0,1,None,NaN,None


In [3]:
five_min_start = pd.Timestamp('2021-10-01 00:00')
aemo_30min = aemo[aemo['SETTLEMENTDATE'] <= five_min_start]
aemo_5min = aemo[aemo['SETTLEMENTDATE'] > five_min_start]
print('30-minute:', interval_gaps(aemo_30min['SETTLEMENTDATE'], '30min'))
print('5-minute: ', interval_gaps(aemo_5min['SETTLEMENTDATE'], '5min'))

30-minute: {'expected': 4416, 'actual': 4416, 'duplicates': 0, 'missing': 0, 'off_grid': 0, 'missing_examples': [], 'off_grid_examples': []}
5-minute:  {'expected': 289152, 'actual': 289152, 'duplicates': 0, 'missing': 0, 'off_grid': 0, 'missing_examples': [], 'off_grid_examples': []}


## Reconcile against the local xlsx

The xlsx stamps interval end minus one second, so I round up to the minute. I reconcile from 1 Oct 2021 only.

In [4]:
xlsx = pd.read_excel('data/raw/NSW Price and Demand 20210701_20240630.xlsx')
print('seconds values:', pd.to_datetime(xlsx['SETTLEMENTDATE']).dt.second.unique())
xlsx['SETTLEMENTDATE'] = pd.to_datetime(xlsx['SETTLEMENTDATE']).dt.ceil('min')
print(interval_gaps(xlsx['SETTLEMENTDATE'], '5min'))
xlsx_5min = xlsx[xlsx['SETTLEMENTDATE'] > five_min_start]
reconcile(aemo_5min, xlsx_5min, 'SETTLEMENTDATE', {'RRP': 'NSW1_Price', 'TOTALDEMAND': 'NSW1_Demand'})

seconds values: [59]
{'expected': 315648, 'actual': 315646, 'duplicates': 0, 'missing': 2, 'off_grid': 0, 'missing_examples': [Timestamp('2023-12-05 13:45:00'), Timestamp('2023-12-05 13:50:00')], 'off_grid_examples': []}


,left,right,only_left,only_right,matched,mismatches,max_abs_diff
0,RRP,NSW1_Price,2,0,289150,0,0.005
1,TOTALDEMAND,NSW1_Demand,2,0,289150,0,0.000


Before 1 Oct 2021 the xlsx holds 5-minute dispatch prices. Their half-hour average should equal AEMO's 30-minute trading price.

In [5]:
xlsx_pre = xlsx[xlsx['SETTLEMENTDATE'] <= five_min_start].copy()
xlsx_pre['half_hour_end'] = xlsx_pre['SETTLEMENTDATE'].dt.ceil('30min')
half_hour_mean = xlsx_pre.groupby('half_hour_end')['NSW1_Price'].mean()
trading_price = aemo_30min.set_index('SETTLEMENTDATE')['RRP']
(trading_price - half_hour_mean.reindex(trading_price.index)).abs().max()

np.float64(0.005000000000009663)

## Spot prices in the analysis pool

In [6]:
prices = pd.read_parquet('data/processed/nsw1_prices_5min.parquet')
spot = prices['spot_price']
print(spot.describe(percentiles=[.01, .05, .5, .95, .99, .999]).round(2))
print('negative share:', round((spot < 0).mean(), 4))
print('at floor (<= -999):', (spot <= -999).sum(), ' at or above 15,000:', (spot >= 15000).sum())
print('IQR outlier share:', round(iqr_outliers(spot).mean(), 3))

count    289152.00
mean        130.69
std         387.29
min       -1000.00
1%          -48.76
5%            0.00
50%          89.40
95%         316.44
99%         541.08
99.9%      2114.11
max       16600.00
Name: spot_price, dtype: float64
negative share: 0.0439
at floor (<= -999): 10  at or above 15,000: 59
IQR outlier share: 0.113


The IQR rule flags about one interval in nine. These are real price spikes, and spikes are the point of the question, so I keep them all.

In [7]:
prices.groupby('fy')['spot_price'].mean().round(2)

fy
2022    150.66
2023    144.96
2024    101.57
Name: spot_price, dtype: float64

## June 2022 administered pricing

AEMO applied a $300/MWh cap in NSW from 13 June 2022 and suspended the market from 15 to 24 June. I keep these real settlement prices and flag the period.

In [8]:
at_300 = prices[prices['spot_price'] == 300]
print('intervals at exactly $300 by month:')
print(at_300.groupby(at_300['interval_start'].dt.to_period('M')).size())
print('flagged administered intervals:', int(prices['administered_period'].sum()))

intervals at exactly $300 by month:
interval_start
2022-02       1
2022-05     145
2022-06    1317
2022-07     151
2024-05       2
2024-06       2
Freq: M, dtype: int64
flagged administered intervals: 3336


## Solar farm output

In [9]:
solar_raw = pd.read_excel('data/raw/Solar Farm Dispatch Data.xlsx')
solar_raw['SETTLEMENTDATE'] = pd.to_datetime(solar_raw['SETTLEMENTDATE']).dt.ceil('min')
print(interval_gaps(solar_raw['SETTLEMENTDATE'], '5min'))
profile_table(solar_raw)

{'expected': 315648, 'actual': 315646, 'duplicates': 0, 'missing': 2, 'off_grid': 0, 'missing_examples': [Timestamp('2023-12-05 13:45:00'), Timestamp('2023-12-05 13:50:00')], 'off_grid_examples': []}


,column,dtype,nulls,null_pct,distinct,min,median,max
0,SETTLEMENTDATE,datetime64[us],0,0.0,315646,2021-07-01 00:05:00,NaN,2024-07-01 00:00:00
1,Solar Farm Output,float64,0,0.0,66362,0.0,0.0,102.0


In [10]:
solar_mw = solar_raw.set_index(solar_raw['SETTLEMENTDATE'] - pd.Timedelta('5min'))['Solar Farm Output']
print('zero share:', round((solar_mw == 0).mean(), 3))
print('output 22:00-04:00 above 0.5 MW:', int(((solar_mw > 0.5) & solar_mw.index.hour.isin([22, 23, 0, 1, 2, 3])).sum()))
fy = solar_mw.index.to_period('Q-JUN').qyear
print('capacity factor by FY:', (solar_mw.groupby(fy).mean() / 102).round(3).to_dict())

zero share: 0.531
output 22:00-04:00 above 0.5 MW: 0
capacity factor by FY: {2022: 0.24, 2023: 0.243, 2024: 0.249}


Days where midday output never reaches 5 MW look like outages. I keep them, because a PPA with one farm carries that farm's outages.

In [11]:
midday = solar_mw[(solar_mw.index.hour >= 10) & (solar_mw.index.hour < 15)]
midday_max = midday.groupby(midday.index.date).max()
outage_days = midday_max[midday_max < 5]
print(len(outage_days), 'outage days')
list(outage_days.index)

12 outage days


[datetime.date(2023, 4, 9),
 datetime.date(2023, 4, 27),
 datetime.date(2023, 4, 28),
 datetime.date(2023, 4, 29),
 datetime.date(2023, 4, 30),
 datetime.date(2023, 5, 1),
 datetime.date(2023, 5, 2),
 datetime.date(2023, 5, 3),
 datetime.date(2023, 7, 12),
 datetime.date(2023, 11, 4),
 datetime.date(2023, 11, 5),
 datetime.date(2024, 3, 1)]

## Solar output at negative prices

Negative prices happen on sunny days, so a PPA settled on every interval pays strike minus a negative price exactly when output is highest.

In [12]:
solar = pd.read_csv('data/processed/solar_factor_5min.csv.gz', parse_dates=['interval_start'])
joined = prices.merge(solar, on='interval_start', how='left')
print('unmatched solar rows:', int(joined['solar_factor'].isna().sum()))
midday_join = joined[joined['interval_start'].dt.hour.between(10, 13)]
(midday_join.groupby(midday_join['spot_price'] < 0)['solar_factor'].mean() * 102).round(1).rename('mean_midday_mw')

unmatched solar rows: 0


spot_price
False    76.7
True     89.1
Name: mean_midday_mw, dtype: float64